In [ ]:
!pip install -q kaggle
!kaggle datasets download -d zalando-research/fashionmnist
!unzip fashionmnist.zip -d fashionmnist/
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import tensorflow as tf
# tf.config.optimizer.set_experimental_options({'disable_meta_optimizer': True})
import kagglehub
from matplotlib import pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

ERROR: Operation cancelled by user


In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
train_df = pd.read_csv('fashionmnist/fashion-mnist_train.csv')
test_df = pd.read_csv('fashionmnist/fashion-mnist_test.csv')

In [ ]:
def image_from_row(row, df=train_df):
    return 2*df.iloc[row, 1:].values.reshape(28,28,1)/255 -1

def images_from_df(indices, df=train_df):
    return 2*df.iloc[indices, 1:].values.reshape(len(indices), 28,28,1)/255 -1

In [ ]:
def residual_block(x, filters):
    shortcut = x

    if x.shape[-1] != filters:
        shortcut = layers.Conv2D(
            filters,
            1,
            padding="same"
        )(shortcut)

    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Conv2D(filters, 3, padding="same")(x)

    x = layers.Add()([x, shortcut])
    x = layers.LeakyReLU(0.2)(x)

    return x

In [ ]:
from tensorflow.keras import models, layers
def criar_critico():
  inputs = layers.Input((28,28,1))
  x = layers.Conv2D(64, 3, padding="same")(inputs)
  x = residual_block(x, 64)
  x = residual_block(x, 128)
  x = residual_block(x, 256)
  x = layers.GlobalAveragePooling2D()(x)
  x = layers.Dense(128)(x)
  outputs = layers.Dense(1)(x)
  return models.Model(inputs, outputs)
criar_critico().summary()

In [ ]:
GENERATOR_LATENT_DIM = 512
def criar_gerador():
  inputs = layers.Input((GENERATOR_LATENT_DIM,))
  x = layers.Dense(512)(inputs)
  x = layers.Reshape((4,4,32))(x)
  x = residual_block(x, 128)
  x = layers.UpSampling2D((2,2))(x)
  x = layers.BatchNormalization()(x)
  x = layers.UpSampling2D((2,2))(x)
  x = residual_block(x, 64)
  x = layers.UpSampling2D((2,2))(x)
  x = layers.BatchNormalization()(x)
  x = residual_block(x, 32)
  x = residual_block(x, 16)
  outputs = layers.Conv2D(1, (5, 5), activation='tanh')(x)
  return  models.Model(inputs, outputs)
criar_gerador().summary()

In [ ]:
train_images = train_df.iloc[:, 1:].values.astype("float32")
train_images = train_images.reshape(-1, 28, 28, 1)

train_images = train_images / 127.5 - 1
def images_from_array(indices):
    return train_images[indices]


In [ ]:
# @tf.function
def train_critic(
    imagens_reais,
    optimizer_critico,
    batch_size,
    lambda_gp
  ):
  print("Entrou no train_critic")
  batch = tf.shape(imagens_reais)[0]
  z = tf.random.normal((batch, GENERATOR_LATENT_DIM), dtype=tf.float32)
  imagens_geradas = tf.stop_gradient(
      gerador(z, training=False)
  )
  with tf.GradientTape() as tape:
      y_pred_gerado = critico(imagens_geradas, training=True)
      # print("do y_pred_gerado contain nan?", tf.reduce_any(tf.math.is_nan(y_pred_gerado)))
      y_pred_real = critico(imagens_reais, training=True)
      y_pred_real = tf.debugging.check_numerics(y_pred_real, "critic_real")
      # tf.print(
      #     "critic output",
      #     tf.reduce_min(y_pred_real),
      #     tf.reduce_max(y_pred_real)
      # )
      # print("do y_pred_real contain nan?", tf.reduce_any(tf.math.is_nan(y_pred_real)))
      # print("wasserstein =", tf.reduce_mean(y_pred_gerado) - tf.reduce_mean(y_pred_real))
      epsilon = tf.random.uniform(
          [batch,1,1,1], dtype=tf.float32
      )
      # print("epsilon", epsilon)
      x_hat = (
          epsilon*imagens_reais
          +
          (one-epsilon)*imagens_geradas
      )
      with tf.GradientTape() as gp_tape:
          gp_tape.watch(x_hat)

          # with tf.device('/CPU:0'):
          pred = critico(x_hat, training=True)
      grad = gp_tape.gradient(pred, x_hat)
      del gp_tape
      grad = tf.debugging.check_numerics(grad, "gp_grad")
      norm = tf.sqrt(tf.reduce_sum(tf.square(grad), axis=[1,2,3]) + 1e-12)
      norm = tf.debugging.check_numerics(norm, "gp_norm")
      gp = tf.reduce_mean(
          (norm - one) ** 2
      )
      gp = tf.debugging.check_numerics(gp, "gp_value")
      loss_critico = tf.reduce_mean(y_pred_gerado) - tf.reduce_mean(y_pred_real) + lambda_gp*gp
      loss_critico = tf.debugging.check_numerics(loss_critico, "loss_critico")
      # print("loss =", loss_critico)
      # print("gp =", gp)
      # print("norm mean =", tf.reduce_mean(norm))
      # print("norm max =", tf.reduce_max(norm))
  grads_critico = tape.gradient(loss_critico, critico.trainable_variables)
  max_grad_abs = tf.reduce_max([tf.reduce_max(tf.abs(g)) for g in grads_critico])

  # grads_critico = [
  #     tf.debugging.check_numerics(g, f"grad_{v.name}")
  #     for g, v in zip(grads_critico, critico.trainable_variables)
  # ]

  grads_critico, _ = tf.clip_by_global_norm(grads_critico, 5.0)
  # global_norm = tf.debugging.check_numerics(global_norm, "global_norm")

  # grads_critico = [
  #    tf.debugging.check_numerics(g, f"post_clip_grad_{v.name}")
  #    for g, v in zip(grads_critico, critico.trainable_variables)
  # ]
  # for var in optimizer_critico.variables:
  #   print(var.name, var.shape)

  # kernel_var = critico.trainable_variables[0]  # ou identifique o "kernel" exato que corrompe
  # kernel_norms = [tf.norm(v) for v in critico.trainable_variables]

  optimizer_critico.apply_gradients(zip(grads_critico, critico.trainable_variables))
  # any_nan = tf.reduce_any([tf.reduce_any(tf.math.is_nan(v)) for v in critico.trainable_variables])

  #tf.debugging.check_numerics(kernel_var, "kernel_apos_update")
  #tf.print("kernel antes tinha nan?", tf.reduce_any(tf.math.is_nan(antes)))
  #tf.print("kernel depois tinha nan?", tf.reduce_any(tf.math.is_nan(kernel_var)))
  #tf.print("grad clipado max abs:", tf.reduce_max(tf.abs(grads_critico[0])))
  # checks = []
  # for var, m, v_slot in zip(
  #   critico.trainable_variables,
  #   optimizer_critico._momentums,
  #   optimizer_critico._velocities,
  # ):
  #   checks.append(tf.debugging.check_numerics(m, f"adam_m_{var.name}"))
  #   checks.append(tf.debugging.check_numerics(v_slot, f"adam_v_{var.name}"))

  # with tf.control_dependencies(checks):
  #   imagens_reais = tf.identity(imagens_reais)  # força a execução de todos os checks acima
  for i, v in enumerate(critico.trainable_variables):
    if tf.reduce_any(tf.math.is_nan(v)):
        print(f"Critico corrompeu: index={i}, name={v.name}, path={v.path}, shape={v.shape}")
        raise RuntimeError("NaN no crítico")
  # return global_norm, max_grad_abs, kernel_norms, any_nan

# @tf.function
def train_generator(
    imagens_reais,
    optimizer_gerador,
    batch_size,
    lambda_gp,
    tamanho_batch_atual
):
  batch = tf.shape(imagens_reais)[0]
  z = tf.random.normal((tamanho_batch_atual, GENERATOR_LATENT_DIM), dtype=tf.float32)
  with tf.GradientTape() as tape:
      imagens_geradas = gerador(z, training=True)
      y_pred_gerado = critico(imagens_geradas, training=False)
      loss_gerador = -tf.reduce_mean(y_pred_gerado)

  grads_gerador = tape.gradient(loss_gerador, gerador.trainable_variables)
  optimizer_gerador.apply_gradients(
      zip(grads_gerador, gerador.trainable_variables)
  )
  for i, v in enumerate(gerador.trainable_variables):
    if tf.reduce_any(tf.math.is_nan(v)):
        print(f"Gerador corrompeu: index={i}, name={v.name}, path={v.path}, shape={v.shape}")
        raise RuntimeError("NaN no gerador")


In [ ]:
import math
from tqdm.notebook import tqdm
from keras.losses import BinaryCrossentropy
from keras.optimizers import Adam
import pickle
import time


batch_size = 300
epochs = 50
lambda_gp = 10
num_batches = int(math.ceil(train_df.shape[0]/batch_size))
n_critic = 5
save_interval = 1
dataset = tf.data.Dataset.from_tensor_slices(train_images)
dataset = dataset.shuffle(
    len(train_images)
).batch(
    batch_size
).prefetch(
    tf.data.AUTOTUNE
)
# tf.config.run_functions_eagerly(True)
# tf.debugging.enable_check_numerics()
# tf.config.experimental.enable_tensor_float_32_execution(False)
# tf.config.experimental.enable_op_determinism()
# tf.random.set_seed(1)
if tf.config.list_physical_devices('GPU'):
  device = '/device:GPU:0'
else:
  device = '/device:CPU:0'
with tf.device(device):
  gerador = criar_gerador()
  critico = criar_critico()
  optimizer_gerador = Adam(learning_rate=0.0001, beta_1=0.0, beta_2=0.9)
  optimizer_critico = Adam(learning_rate=0.0001, beta_1=0.0, beta_2=0.9)
  # WARMING UP THE OPTIMIZERS
  dummy = tf.zeros((1, 28, 28, 1))

  with tf.GradientTape() as tape:
      out = critico(dummy)

  grads = tape.gradient(out, critico.trainable_variables)

  optimizer_critico.apply_gradients(
      zip(grads, critico.trainable_variables)
  )
  z = tf.random.normal((10, GENERATOR_LATENT_DIM), dtype=tf.float32)
  with tf.GradientTape() as tape:
      imagens_geradas = gerador(z, training=True)
      y_pred_gerado = critico(imagens_geradas, training=False)
      loss_gerador = -tf.reduce_mean(y_pred_gerado)

  grads_gerador = tape.gradient(loss_gerador, gerador.trainable_variables)
  optimizer_gerador.apply_gradients(
      zip(grads_gerador, gerador.trainable_variables)
  )

  # CHECKPOINTS DEFINITION AND RESTORATION
  ckpt = tf.train.Checkpoint(
      gerador=gerador,
      critico=critico,
      optimizer_gerador=optimizer_gerador,
      optimizer_critico=optimizer_critico,
      epoch=tf.Variable(0),
      batch=tf.Variable(0)
  )

  manager = tf.train.CheckpointManager(
      ckpt,
      "drive/MyDrive/machine_learning_systems_output/Fashion_MNIST/checkpoints",
      max_to_keep=1
  )
  print(manager.latest_checkpoint)
  ckpt.restore(manager.latest_checkpoint)

  initial_epoch = 0
  initial_batch = 0
  if manager.latest_checkpoint:
      print("Checkpoint carregado!")
      print("Época:", int(ckpt.epoch))
      print("Batch:", int(ckpt.batch))
      initial_epoch =  int(ckpt.epoch.numpy())
      initial_batch = int(ckpt.batch.numpy())
  else:
      print("Treinamento do zero.")
  one = tf.constant(1.0, dtype=tf.float32)
  for epoch in tqdm(
      range(initial_epoch, epochs),
      initial=initial_epoch,
      total=epochs,
      desc="epoch"
  ):
      start_batch = initial_batch if epoch == initial_epoch else 0
      dataset_epoch = dataset.skip(start_batch)
      for batch_index, imagens_reais in tqdm(
          enumerate(dataset_epoch, start=start_batch),
          initial=start_batch,
          total=num_batches,
          desc="batch",
          leave=False
      ):
          if batch_index < start_batch:
              continue

          tamanho_batch_atual = imagens_reais.shape[0]
          ruido = tf.random.normal(
              imagens_reais.shape,
              stddev=0.05,
              dtype=imagens_reais.dtype
          )
          imagens_reais += ruido
          imagens_reais = tf.clip_by_value(
              imagens_reais,
              -1.0,
              1.0
          )
          for _ in range(n_critic):
            train_critic(imagens_reais, optimizer_critico, batch_size, lambda_gp)
            # print("global_norm:", gn.numpy(), "max_grad_abs:", mg.numpy(), "kernel norm antes:", kernel_norm_antes, "kernel nan?:", corrompeu.numpy())
          train_generator(
              imagens_reais,
              optimizer_gerador,
              batch_size,
              lambda_gp,
              tamanho_batch_atual
          )
          if batch_index % save_interval == 0:
            if batch_index == num_batches - 1:
              ckpt.epoch.assign(epoch + 1)
              ckpt.batch.assign(0)
            else:
              ckpt.epoch.assign(epoch)
              ckpt.batch.assign(batch_index + 1)
            save_path = manager.save()
            print("Checkpoint salvo em:", save_path)

**When the training needs to be stopped run the cells bellow to save the checkpoints and results on the repository**

In [ ]:
os.makedirs('drive/MyDrive/machine_learning_systems_output/Fashion_MNIST/generator_outputs', exist_ok=True)
z = z = tf.random.normal((10, GENERATOR_LATENT_DIM))
imagens = gerador(z)
print(imagens.shape)
print(imagens.dtype)

print(tf.reduce_min(imagens))
print(tf.reduce_max(imagens))

print(tf.math.reduce_any(tf.math.is_nan(imagens)))
print(tf.math.reduce_any(tf.math.is_inf(imagens)))
for i in range(10):
    plt.figure()
    plt.imsave(f'drive/MyDrive/machine_learning_systems_output/Fashion_MNIST/generator_outputs/{i}.png', tf.squeeze(imagens[i]), cmap="gray")

In [ ]:
plt.imshow(tf.squeeze(imagens[0].numpy()), cmap="gray")
plt.colorbar()
plt.show()

In [ ]:
z = tf.random.normal((10, GENERATOR_LATENT_DIM))
imgs = gerador(z, training=False)

print(tf.reduce_min(imgs))
print(tf.reduce_max(imgs))
print(tf.math.reduce_any(tf.math.is_nan(imgs)))

In [ ]:
for v in gerador.trainable_variables:
    if tf.reduce_any(tf.math.is_nan(v)):
        print(v.name, "contém NaN")

In [ ]:
gerador = criar_gerador()
z = tf.random.normal((10, GENERATOR_LATENT_DIM))
imgs = gerador(z, training=False)

print(tf.reduce_min(imgs))
print(tf.reduce_max(imgs))
print(tf.math.reduce_any(tf.math.is_nan(imgs)))